# IEEE-CIS Fraud Detection — საბაზისო EDA

ეს notebook-ი აანალიზებს IEEE-CIS Fraud Detection-ის train მონაცემებს.

**მიზანი:** გავიგოთ feature-ების ბუნება, missing values-ის სტრუქტურა, target-ის distribution და ძირითადი კავშირები fraud-თან.

## 1. Setup

In [ ]:
# საჭირო ბიბლიოთეკების იმპორტი
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

FRAUD_PALETTE = {0: "#4C72B0", 1: "#C44E52"}  # ლეგიტიმური — ცისფერი, fraud — წითელი


## 2. მონაცემების ჩატვირთვა

In [ ]:
KAGGLE_PATH = "/kaggle/input/ieee-fraud-detection"
LOCAL_PATH  = "./data"
DATA_PATH   = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

train_tx = pd.read_csv(f"{DATA_PATH}/train_transaction.csv")
train_id = pd.read_csv(f"{DATA_PATH}/train_identity.csv")
print("train_transaction:", train_tx.shape)
print("train_identity   :", train_id.shape)

# ვაერთებთ identity ფაილს transaction-ზე
train = train_tx.merge(train_id, on="TransactionID", how="left")
print("merged           :", train.shape)
del train_tx, train_id


## 3. Target-ის Distribution

Fraud detection ამოცანის მთავარი პრობლემა — class imbalance. ვამოწმებთ რამდენად disbalanced-ია ჩვენი დატა.

In [ ]:
fraud_rate = train["isFraud"].mean()
print(f"Fraud rate: {fraud_rate:.4%}  -> ძლიერი class imbalance")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
train["isFraud"].value_counts().plot(kind="bar", ax=ax[0],
    color=[FRAUD_PALETTE[0], FRAUD_PALETTE[1]])
ax[0].set_title("კლასების რაოდენობა"); ax[0].set_xticklabels(["Legit", "Fraud"], rotation=0)

train["isFraud"].value_counts(normalize=True).plot(kind="bar", ax=ax[1],
    color=[FRAUD_PALETTE[0], FRAUD_PALETTE[1]])
ax[1].set_title("კლასების პროპორცია"); ax[1].set_xticklabels(["Legit", "Fraud"], rotation=0)
plt.tight_layout(); plt.show()


## 4. Missing Values-ის ანალიზი

IEEE-CIS დატაში ბევრ feature-ს აქვს მაღალი NaN rate. ვამოწმებთ რამდენი კოლონაა > 50% / > 90% ცარიელი.

In [ ]:
missing = train.isna().mean().sort_values(ascending=False)
print(f"სულ კოლონები: {len(missing)}")
print(f">50% NaN: {(missing > 0.5).sum()}")
print(f">90% NaN: {(missing > 0.9).sum()}")

plt.figure(figsize=(11, 4))
missing.head(40).plot(kind="bar", color="steelblue")
plt.title("ტოპ-40 კოლონა missing rate-ის მიხედვით")
plt.ylabel("NaN-ის წილი"); plt.tight_layout(); plt.show()


## 5. TransactionAmt-ის Distribution

TransactionAmt არის ერთ-ერთი მთავარი feature. ის ძლიერად skewed-ია, ამიტომ log-ტრანსფორმაცია ხშირად ეხმარება.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
train["TransactionAmt"].clip(0, 1000).hist(bins=80, ax=ax[0], color="steelblue")
ax[0].set_title("TransactionAmt (clipped 1000-ზე)")

np.log1p(train["TransactionAmt"]).hist(bins=80, ax=ax[1], color="seagreen")
ax[1].set_title("log(1 + TransactionAmt)")
plt.tight_layout(); plt.show()

# Fraud vs non-fraud-ის შედარება
print("\nFraud vs Legit საშუალო ტრანზაქცია:")
print(train.groupby("isFraud")["TransactionAmt"].describe()[["mean", "50%", "max"]])


## 6. კატეგორიული Feature-ები

ვაჩვენებთ თითოეულ კატეგორიაში fraud rate-ს რომ ნახოთ რომელი value-ები ასოცირდება მაღალ რისკთან.

In [ ]:
cat_cols = ["ProductCD", "card4", "card6", "P_emaildomain", "DeviceType"]

for col in cat_cols:
    if col not in train.columns:
        continue
    plt.figure(figsize=(8, 3))
    fr = train.groupby(col)["isFraud"].agg(["mean", "count"]).sort_values("count", ascending=False).head(10)
    fr["mean"].plot(kind="bar", color="tomato")
    plt.title(f"Fraud rate by {col} (ტოპ-10 count-ით)")
    plt.ylabel("Fraud rate"); plt.tight_layout(); plt.show()


## 7. რიცხვითი Feature-ების კორელაცია Target-თან

In [ ]:
num_cols = train.select_dtypes(include=np.number).columns.drop("isFraud")
corr = train[num_cols].corrwith(train["isFraud"]).abs().sort_values(ascending=False).head(25)

plt.figure(figsize=(8, 6))
corr.plot(kind="barh", color="darkblue")
plt.title("ტოპ-25 რიცხვითი ფიჩერი Target-თან კორელაციით")
plt.tight_layout(); plt.show()


## 8. დროითი ანალიზი (TransactionDT)

In [ ]:
if "TransactionDT" in train.columns:
    train["hour"] = (train["TransactionDT"] // 3600) % 24
    plt.figure(figsize=(11, 3))
    train.groupby("hour")["isFraud"].mean().plot(kind="bar", color="tomato")
    plt.title("Fraud rate დღის საათის მიხედვით")
    plt.ylabel("Fraud rate"); plt.tight_layout(); plt.show()


## 9. დასკვნები

- დატა ძლიერ disbalanced-ია (≈3.5% fraud), რაც ნიშნავს რომ აუცილებელია   `class_weight=balanced` ან sampling ტექნიკა.
- ბევრ feature-ს აქვს მაღალი missing rate (>50%) — საჭიროა cleaning + imputation.
- TransactionAmt skewed-ია — log-ტრანსფორმაცია სტაბილიზებს distribution-ს.
- კატეგორიულ feature-ებს (ProductCD, card4, P_emaildomain) აქვთ   მნიშვნელოვნად განსხვავებული fraud rate-ები.
- რამდენიმე V-/C- ფიჩერი ძლიერად კორელირებს target-თან.

შემდეგი ნაბიჯი: ცალცალკე notebook-ი თითო მოდელის არქიტექტურისთვის.